In [1]:
!git clone https://github.com/ag-kunal/news-bias-detection.git
%cd news-bias-detection
!pip install -q -r requirements.txt
!mkdir -p fallacy_detection/reference_data
%cd fallacy_detection/reference_data
!git clone -q --depth 1 https://github.com/ChadiHelwe/MAFALDA
!git clone -q --depth 1 https://github.com/pierpaologoffredo/FallacyDetection
!git clone -q --depth 1 https://github.com/causalNLP/logical-fallacy
%cd ../..
!python fallacy_detection/prepare_fallacy_data.py

Cloning into 'news-bias-detection'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 152 (delta 24), reused 138 (delta 14), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 3.27 MiB | 16.98 MiB/s, done.
Resolving deltas: 100% (24/24), done.
/kaggle/working/news-bias-detection
/kaggle/working/news-bias-detection/fallacy_detection/reference_data
/kaggle/working/news-bias-detection
MAFALDA: 837 rows
ElecDeb: 5,091 rows
LOGIC: 3,800 rows

Saved 8,057 rows -> /kaggle/working/news-bias-detection/outputs/fallacy_detection/training/fallacy_sentences.csv
source         is_fallacy
elecdeb        0.0           2273
               1.0           1447
logic_climate  NaN           1075
logic_edu      NaN           2425
mafalda        0.0            529
               1.0            308

Type counts (positives with a harmonised type):
source                     elecdeb  logic_climate  log

In [2]:
%%writefile fallacy_detection/finetune_binary.py
# finetune_binary.py
# Fine-tune RoBERTa for binary sentence-level fallacy detection.
# Usage:
#   python fallacy_detection/finetune_binary.py --csv PATH --holdout debates
#   python fallacy_detection/finetune_binary.py --csv PATH --pooled
import argparse, numpy as np, pandas as pd, torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score
from sklearn.model_selection import GroupShuffleSplit

def gsplit(d, frac, seed):
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
                .split(d, groups=d[GROUP_COL]))
    return d.iloc[a], d.iloc[b]

# ---- EDIT THESE after looking at the 1A output ----
TEXT_COL, LABEL_COL, SOURCE_COL, GROUP_COL = "text", "is_fallacy", "source", "group"
# ----------------------------------------------------

def to_binary(x):
    if isinstance(x, str):
        return 0 if x.strip().lower() in {"0", "none", "nothing", "no fallacy", "no_fallacy", "false"} else 1
    return int(x)

ap = argparse.ArgumentParser()
ap.add_argument("--csv", required=True)
ap.add_argument("--model", default="roberta-base")
ap.add_argument("--holdout", default=None, help="source name to hold out as test set")
ap.add_argument("--pooled", action="store_true", help="random 80/20 split instead")
ap.add_argument("--epochs", type=int, default=3)
ap.add_argument("--bs", type=int, default=16)
ap.add_argument("--lr", type=float, default=2e-5)
ap.add_argument("--max_len", type=int, default=128)
ap.add_argument("--seed", type=int, default=42)
ap.add_argument("--save", default=None, help="dir to save model+tokenizer")
args = ap.parse_args()

torch.manual_seed(args.seed); np.random.seed(args.seed)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev)

df = pd.read_csv(args.csv).dropna(subset=[TEXT_COL, LABEL_COL, GROUP_COL])
df["y"] = df[LABEL_COL].map(to_binary)
print(df.groupby(SOURCE_COL)["y"].agg(["count", "mean"]))

if args.holdout:
    test = df[df[SOURCE_COL] == args.holdout]
    train = df[df[SOURCE_COL] != args.holdout]
    assert len(test) and len(train), "bad --holdout name"
else:
    train, test = gsplit(df, 0.2, args.seed)
train, val = gsplit(train, 0.1, args.seed)
print(f"train {len(train)} | val {len(val)} | test {len(test)}")

tok = AutoTokenizer.from_pretrained(args.model)
model = AutoModelForSequenceClassification.from_pretrained(args.model, num_labels=2).to(dev)

def make_loader(d, shuffle):
    texts, ys = d[TEXT_COL].astype(str).tolist(), d["y"].tolist()
    def collate(idx):
        enc = tok([texts[i] for i in idx], truncation=True, max_length=args.max_len,
                  padding=True, return_tensors="pt")
        enc["labels"] = torch.tensor([ys[i] for i in idx])
        return enc
    return DataLoader(list(range(len(texts))), batch_size=args.bs, shuffle=shuffle, collate_fn=collate)

tr_dl, va_dl, te_dl = make_loader(train, True), make_loader(val, False), make_loader(test, False)

# class-weighted loss for imbalance
pos = train["y"].mean()
w = torch.tensor([1 / (1 - pos), 1 / pos], dtype=torch.float, device=dev); w = w / w.sum() * 2
loss_fn = torch.nn.CrossEntropyLoss(weight=w)

opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=0.01)
steps = len(tr_dl) * args.epochs
sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
scaler = torch.cuda.amp.GradScaler(enabled=(dev == "cuda"))

@torch.no_grad()
def predict(dl):
    model.eval(); ps, ys = [], []
    for b in dl:
        b = {k: v.to(dev) for k, v in b.items()}
        y = b.pop("labels")
        with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
            logits = model(**b).logits
        ps.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy()); ys.append(y.cpu().numpy())
    return np.concatenate(ps), np.concatenate(ys)

def report(p, y, thr=0.5, tag=""):
    pred = (p >= thr).astype(int)
    print(f"{tag} AUC {roc_auc_score(y, p):.3f} | F1 {f1_score(y, pred):.3f} | macroF1 {f1_score(y, pred, average='macro'):.3f} "
          f"| P {precision_score(y, pred, zero_division=0):.3f} | R {recall_score(y, pred):.3f} | flagged {pred.mean():.2%}")

best_auc, best_state = -1, None
for ep in range(args.epochs):
    model.train()
    for b in tr_dl:
        b = {k: v.to(dev) for k, v in b.items()}
        y = b.pop("labels")
        with torch.autocast(device_type=dev, enabled=(dev == "cuda")):
            logits = model(**b).logits
        loss = loss_fn(logits.float(), y)
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update(); sched.step()
    p, y = predict(va_dl); report(p, y, tag=f"epoch {ep+1} val")
    auc = roc_auc_score(y, p)
    if auc > best_auc:
        best_auc = auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
p, y = predict(te_dl)
print("\n=== TEST", f"(held-out: {args.holdout})" if args.holdout else "(pooled random split)", "===")
report(p, y, tag="test")
print(f"base rate of fallacy in test: {y.mean():.2%}")

if args.save:
    model.save_pretrained(args.save); tok.save_pretrained(args.save)
    print("saved to", args.save)

Writing fallacy_detection/finetune_binary.py


In [7]:
%cd /kaggle/working/news-bias-detection

[Errno 2] No such file or directory: '/kaggle/working/news-bias-detection'
/news-bias-detection


In [5]:
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --pooled
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --holdout elecdeb
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --holdout mafalda


model.safetensors: downloading bytes: █████▊                |  130MB, 8.90MB/s  epoch 2 val AUC 0.900 | F1 0.766 | macroF1 0.799 | P 0.733 | R 0.803 | flagged 43.63%
epoch 3 val AUC 0.902 | F1 0.777 | macroF1 0.812 | P 0.760 | R 0.796 | flagged 41.73%

=== TEST (pooled random split) ===
test AUC 0.899 | F1 0.759 | macroF1 0.801 | P 0.727 | R 0.794 | flagged 41.11%
base rate of fallacy in test: 37.67%
epoch 1 val AUC 0.812 | F1 0.533 | macroF1 0.650 | P 0.432 | R 0.696 | flagged 41.11%
device: cuda
         count      mean
source                  
elecdeb   3720  0.388978
mafalda    837  0.367981
train 3350 | val 370 | test 837
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2592.20it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNE

In [7]:
!python fallacy_detection/finetune_binary.py --csv outputs/fallacy_detection/training/fallacy_sentences.csv --pooled --save outputs/fallacy_detection/roberta_binary

device: cuda
         count      mean
source                  
elecdeb   3720  0.388978
mafalda    837  0.367981
train 3288 | val 369 | test 900
Loading weights: 100%|██████████████████████| 197/197 [00:00<00:00, 2706.20it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training